# Prinzessin der Verurteilung — a persona-driven chat app

**EECE 798S — Assignment 2**
Laure Mohsen

A web chat app built on an open-source LLM (Llama 3.1 8B), with a two-voice
persona twist: **Fischl** answers in character, and her raven **Oz** says what
she actually meant.

Repository: [github.com/lauremhsn/fischl-oz-chatbot](https://github.com/lauremhsn/fischl-oz-chatbot)

---

**This notebook runs in two places.**

* **Google Colab** — run every cell top to bottom. It installs Ollama, pulls
  the model, and gives you a public link to the running app. Nothing needs to
  be installed on your own machine. A GPU runtime is strongly recommended
  (*Runtime → Change runtime type → T4 GPU*); on CPU it works but is slow.
* **Locally** — if Ollama is already running, the setup cells detect that and
  skip straight past the install.

The model is open-source and self-hosted either way. No API key is involved
and nothing is sent to a third-party service.

## 1. Setup

### 1.1 Python dependencies

In [ ]:
%pip install -q gradio openai
print("dependencies installed")

### 1.2 Ollama

Ollama serves the model behind an OpenAI-compatible endpoint at
`http://localhost:11434/v1`, which is why the app can use the `openai` client
library against a model running on this machine.

This cell is a no-op if Ollama is already up.

In [ ]:
import platform, subprocess, time, urllib.request


def ollama_up(timeout=2):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=timeout)
        return True
    except Exception:
        return False


if ollama_up():
    print("Ollama is already running — nothing to do.")
elif platform.system() == "Linux":
    print("Installing Ollama...")
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh",
                   shell=True, check=True)
    subprocess.Popen(["ollama", "serve"],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(30):
        time.sleep(1)
        if ollama_up():
            break
    print("Ollama running" if ollama_up() else "Ollama did not start — rerun this cell")
else:
    print(f"On {platform.system()}, install Ollama from https://ollama.com/download,")
    print("make sure it is running, then rerun this cell.")

### 1.3 The model

Two steps. First pull the base model, then build a derived model from a
`Modelfile`.

The second step matters more than it looks. Ollama defaults to a **4096**-token
context window and **silently ignores** `num_ctx` passed through the
OpenAI-compatible endpoint, so asking for a longer window at request time does
nothing. Setting it in a `Modelfile` is the only reliable way, and the whole
context-management design below depends on the window actually being 8192.

In [ ]:
import subprocess, textwrap, pathlib

MODELFILE = textwrap.dedent('''
    FROM llama3.1:8b
    PARAMETER num_ctx 8192
''').strip()

pathlib.Path("Modelfile").write_text(MODELFILE)
print(MODELFILE, "\n")

print("Pulling llama3.1:8b (about 4.9 GB — this is the slow step)...")
subprocess.run(["ollama", "pull", "llama3.1:8b"], check=True)
subprocess.run(["ollama", "create", "fischl-llama", "-f", "Modelfile"], check=True)
print("\nfischl-llama built.")

Confirm the context window really is 8192 rather than the 4096 default:

In [ ]:
import subprocess
subprocess.run(["ollama", "run", "fischl-llama", "hi"], capture_output=True)
print(subprocess.run(["ollama", "ps"], capture_output=True, text=True).stdout)

### 1.4 The application code

The modules live in the repository. Cloning rather than pasting them inline
keeps one source of truth — the notebook and the repo cannot drift apart.

In [ ]:
import os, sys, subprocess

# Idempotent: safe to re-run without nesting clones or chdir-ing twice.
if not os.path.exists("chat.py"):
    if not os.path.isdir("fischl-oz-chatbot"):
        subprocess.run(["git", "clone", "https://github.com/lauremhsn/fischl-oz-chatbot"], check=True)
    os.chdir("fischl-oz-chatbot")

if not os.path.exists("chat.py"):
    raise RuntimeError(
        "chat.py not found. If the repository is private or not yet pushed, "
        "the clone above will have failed — push it first, or upload the .py "
        "files into this session."
    )

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print("working directory:", os.getcwd())
print("files:", sorted(f for f in os.listdir() if f.endswith((".py", ".md"))))

---

## 2. The model and its context window

**Model:** Llama 3.1 8B (`llama3.1:8b`), open-weights, served locally.

It was chosen over a hosted API for three reasons: the assignment asks for an
open-source model; an 8B model fits in 8 GB of VRAM at 4-bit quantisation, so
it runs on a laptop GPU; and running it locally means no key, no quota, and no
data leaving the machine.

**Context window: 8192 tokens.** Llama 3.1 supports 128k, but the KV cache is
what decides what actually fits. At roughly 128 KB per token (32 layers × 8 KV
heads × 128 head dim × 2 tensors × 2 bytes), 8192 tokens is about 1 GB of
cache, which sits alongside 4.9 GB of weights inside an 8 GB card.

### What happens as that window fills

Every call is assembled in the same order, and only one part is allowed to
shrink:

```
[ system prompt ][ pinned few-shot ][ conversation history ][ new message ]
  ~350 tokens      ~450 tokens        grows                   varies
```

| Constant | Value | Meaning |
|---|---|---|
| `CONTEXT_LIMIT` | 8192 | the window, matching the Modelfile |
| `FIXED_OVERHEAD` | 850 | measured: system prompt + pinned few-shot |
| `RESERVED_FOR_REPLY` | 512 | held back so there is room to generate into |
| `HISTORY_BUDGET` | 6830 | what is left for history |
| `SUMMARISE_BATCH` | 6 | turns folded per compaction pass |
| `KEEP_VERBATIM` | 4 | most recent turns, never summarised |

A turn costs about **104 tokens**, so roughly **65 turns** fit before anything
is discarded. After that the oldest six are folded into a running summary and
dropped from verbatim history, while the four most recent are never eligible —
so pronouns and follow-ups still resolve against real text.

The few-shot examples are **pinned**: dropping them is what makes a persona
drift, and a naive sliding window over the message list would evict them first.

In [ ]:
import chat

print(f"CONTEXT_LIMIT      {chat.CONTEXT_LIMIT}")
print(f"FIXED_OVERHEAD     {chat.FIXED_OVERHEAD}")
print(f"RESERVED_FOR_REPLY {chat.RESERVED_FOR_REPLY}")
print(f"HISTORY_BUDGET     {chat.HISTORY_BUDGET}")
print(f"SUMMARISE_BATCH    {chat.SUMMARISE_BATCH}")
print(f"KEEP_VERBATIM      {chat.KEEP_VERBATIM}")
print(f"CHRONICLE_WORD_CAP {chat.CHRONICLE_WORD_CAP}")

---

## 3. The persona twist

Fischl is a character who speaks entirely in self-declared royal grandeur. Her
raven Oz follows every pronouncement with one dry line telling you what she
actually meant. The twist is that the app is **two model calls with two system
prompts**, not one call asked to produce two voices — and the interface shows
them as two separate bubbles so the chain is visible rather than hidden.

### Prompting techniques used, and why

**1. Few-shot prompting** — `FISCHL_FEWSHOT`, `OZ_EXAMPLES`

The examples do most of the work. They carry register, length range, and the
knowledge boundary without naming any of it.

This was learned the hard way. An earlier version ran to roughly 900 words of
rules for Fischl alone, and the output got steadily *worse* in a consistent
direction: whatever vocabulary appeared in the prompt came back as a tic.
Banning one stock phrase produced a different one in its place; listing
alternative imagery to vary it put those images in every reply; forbidding the
model to mention a "chronicle" taught it the word, and it announced one in turn
one, before any chronicle existed.

A small model does not weigh a long instruction list evenly. It fixates on
salient terms, and **a prohibition is a mention**. So the prompts state the
minimum, name as few specific phrases as possible, and let examples demonstrate
what a prohibition cannot.

**2. Prompt chaining** — `FISCHL_SYSTEM` → `OZ_SYSTEM`

Oz's reply is a second call that consumes Fischl's output. He deliberately does
*not* receive the conversation history: given the backlog he summarises the
conversation instead of answering the turn in front of him.

**3. Chain-of-thought** — `REASONING_SYSTEM` (the reasoning-mode toggle)

Covered in section 6.

In [ ]:
from persona import FISCHL_SYSTEM, FISCHL_FEWSHOT, OZ_SYSTEM, OZ_EXAMPLES

print(FISCHL_SYSTEM)
print("\n" + "=" * 70)
print(f"{len(FISCHL_FEWSHOT) // 2} few-shot pairs for Fischl, "
      f"{len(OZ_EXAMPLES) // 2} for Oz\n")
for m in FISCHL_FEWSHOT[:4]:
    print(f"  {m['role']:>9}: {m['content'][:90]}")

---

## 4. One turn through the chain

The user message goes to Fischl with the system prompt, the pinned examples and
the history; her reply is then handed to Oz as a second call.

In [ ]:
from chat import Conversation, respond

conversation = Conversation()
turn = respond(conversation, "Hello! What's 17 times 23?")

print("YOU    :", turn.user)
print()
print("FISCHL :", turn.fischl)
print()
print("OZ     :", turn.oz)

---

## 5. Context handling in action

At the real budget the window holds about 65 turns, which is more than a demo
can show. Lowering `HISTORY_BUDGET` forces the same machinery to run after a
few exchanges — the app exposes this as a "Demo controls" box for the same
reason.

Watch three things: `verbatim` stops growing, the chronicle appears, and the
dossier keeps its facts throughout.

In [ ]:
import chat
from chat import Conversation, respond

chat.HISTORY_BUDGET = 350          # normally 6830
conversation = Conversation()

messages = [
    "Hi! My name is Wren.",
    "I'm building a chat app for a university course.",
    "It's due next Friday and I'm a bit behind.",
    "What's a good way to stop procrastinating?",
    "I also like pizza, for the record.",
    "Do you remember my name?",
    "What am I working on again?",
]

for m in messages:
    turn = respond(conversation, m)
    print(f"YOU    : {m}")
    print(f"FISCHL : {turn.fischl[:140]}")
    print(f"         [verbatim turns: {len(conversation.turns)}  "
          f"history: {conversation.history_tokens()} tok  "
          f"folded this turn: {conversation.last_folded}]")
    print()

print("=" * 70)
print("DOSSIER (pinned, updated every turn):")
print(conversation.facts)
print()
print("CHRONICLE (written only during compaction, capped at "
      f"{chat.CHRONICLE_WORD_CAP} words):")
print(conversation.chronicle or "(still empty)")

chat.HISTORY_BUDGET = 6830         # restore

The two memories do different jobs:

* The **dossier** is small, pinned, and refreshed every turn. It holds stable
  facts the person stated about themselves and is never summarised away.
* The **chronicle** is written only during compaction. It is rewritten rather
  than appended to, and capped at 70 words in code as well as in its prompt.

They are separate because the chronicle demonstrably cannot be trusted to hold
user facts: in testing it kept Fischl's advice while dropping the user's name,
producing a bot that denied knowing a name sitting in its own context.

The word cap is enforced in code because the prompt alone did not hold. Asked
for "80 words maximum", the model treated it as a target to approach and then
drift past — 47 → 87 words over four folds. A summariser that creeps past its
cap defeats the purpose of having one.

---

## 6. Reasoning mode (stretch goal)

With the toggle on, a **plain-voice** pass works the problem out first and ends
with a single `ANSWER:` line. Fischl then announces that conclusion in
character. The interface shows the trace in its own collapsible panel above her
reply, visually separate from the answer.

**Why the reasoning happens outside the persona.** An accuracy harness
(`check_accuracy.py`) found that heavier persona load costs correctness: 17 × 23
came back as "three hundred and eleven", and *every* wrong answer was one the
model had spelled out in words while decorating it. Reasoning in the ornate
voice would put the arithmetic back inside the thing that breaks it. Temperature
is 0.2 here — this stage wants the likeliest next token, not an interesting one.

In [ ]:
from chat import Conversation, fischl_reply, reasoning_stream

QUESTION = ("A train leaves at 14:25 and the journey takes 3 hours and "
            "50 minutes. What time does it arrive?")

print("REASONING OFF")
print("-" * 70)
print(fischl_reply(Conversation().turns, QUESTION))

print("\n\nREASONING ON")
print("-" * 70)
trace = ""
for trace in reasoning_stream(QUESTION):
    pass
print("[trace]\n" + trace)
print("\n[answer]")
print(fischl_reply(Conversation().turns, QUESTION, reasoning=trace))

### Measured effect

`check_reasoning.py` runs six multi-step problems four times each in both
modes. Correct answers are accepted in digit *and* spelled-out form — an
earlier version checked only for digits and scored a correct "twenty-eight" as
a failure, which inflated the apparent benefit of reasoning mode, since the
persona spells numbers out far more often with reasoning off.

| Question | OFF | ON |
|---|---|---|
| Train, 14:25 + 3h50m | 2/4 | **4/4** |
| 5 machines / 5 widgets | 3/4 | **4/4** |
| 12 people shaking hands | 3/4 | **4/4** |
| Shelf arithmetic | 4/4 | 4/4 |
| Age word problem | 4/4 | 4/4 |
| Two successive discounts | 4/4 | 4/4 |
| **Overall** | **20/24 (83%)** | **24/24 (100%)** |

**Two examples where it changes the answer:**

**Train.** With reasoning off: *"Subtract 3 hours, 50 minutes from 14:25… 10:35."*
It subtracted instead of adding. The trace does it in two plain lines and lands
on 18:15.

**Handshakes.** With reasoning off you can watch it come apart mid-sentence:

> *"Eleven times twelve… no, wait, we count not each greeting twice. Eleven
> times twelve, minus the dozen hands I have already counted, and we find…
> sixty, no. Sixty was too soon… and we find… forty-eight."*

That is the persona load eating the arithmetic in real time, and it is the
whole argument for doing the working in a plain voice first.

In [ ]:
# Re-run the full comparison yourself (a few minutes).
# %run check_reasoning.py

---

## 7. Run the app

`share=True` returns a public URL as well as the local one, which is what makes
the app reachable when this notebook is running on a Colab VM. The link lives
for about 72 hours.

In [ ]:
import app

app.demo.launch(
    theme=app.THEME,
    css=app.CSS,
    favicon_path=str(app._FAVICON),
    pwa=True,
    share=True,
)

---

## Notes

**Interface.** Fischl and Oz get separate bubbles because they are separate
calls; Gradio's `group_consecutive_messages` defaults to `True` and silently
merges them back into one, which would hide the chain the app is built around.
Replies stream a word at a time with a typing indicator, and the right-hand
panel shows window usage, the dossier and the chronicle — context management is
invisible until it fails, and a conversation that silently forgets is
indistinguishable from one that never knew.

**Artwork.** The Fischl and Oz avatars were drawn for this project by a friend
and are used with permission. The starfield and the raven constellation are
drawn in SVG in `app.py`. No game assets are used.

**Diagnostics.** `check_ollama.py`, `check_context.py`, `check_persona.py`,
`check_chain.py`, `check_context_mgmt.py`, `check_accuracy.py` and
`check_reasoning.py` are the harnesses behind the numbers quoted above. They
are kept in the repository rather than deleted, because every measured claim in
this notebook is reproducible by running one of them.